# Structured Outputs

LLMs regurgitate out text and that is great for so many applications. But in order to build strong, robust systems and applications, we need to make sense of the chaos sometimes by receiving a pre-determined structured output everytime an LLM is called.

## As always, libraries first!

In [33]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

# check if API keys are set
if not OPENAI_API_KEY:
    raise ValueError("Missing OpenAI API key")


## The Workflow

```mermaid
graph LR
    A[Generate Ticket] --> B[Respond to Ticket]
    B --> C[Evaluate Response]
    C --> B
    C --> D[Final Output]
```

## Introducing the Pydantic library

In [34]:
# classes
from pydantic import BaseModel

class CustomerTicket(BaseModel):
    ticket: str
    priority: str
    assigned_to: str
 
class TicketResponse(BaseModel):
    response: str
    resolution_time: str

class TicketEvaluation(BaseModel):
    passed: bool
    feedback: str

## Calling OpenAI to generate support tickets

In [35]:
# client
client = OpenAI()

In [36]:
# messages list
user_message = "I want you to generate a customer support ticket for a 3rd party tech re-seller. "
user_message += "The ticket should be a single sentence describing a common issue a customer might face with their product or service. "
user_message += "Please ensure the ticket is varied and covers different types of problems."

messages = [{"role": "user", "content": user_message}]

In [37]:
# normal response
response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages
)

normal_response = response.choices[0].message.content
display(Markdown(f"### Normal Response:\n{normal_response}"))

### Normal Response:
The customer is unable to activate the software license purchased through the reseller despite entering the correct activation code.

In [38]:
# structured response, will generate a "random" ticket that fits the CustomerTicket schema
structured_response = client.chat.completions.parse(
    model="gpt-4.1-nano",
    messages=messages,
    # new, specify the response format
    response_format=CustomerTicket
)
# new, access message.parsed instead of message.content
structured_response = structured_response.choices[0].message.parsed
display(Markdown(f"### Structured Response:\n{structured_response}"))

### Structured Response:
ticket='The customer is unable to activate their purchased software due to a licensing error.' priority='High' assigned_to='Technical Support Team'

In [39]:
structured_response.ticket

'The customer is unable to activate their purchased software due to a licensing error.'

In [40]:
structured_response.priority

'High'

## Responding to the ticket

In [41]:
# messages list
message = "You are to propose a resolution for the following customer support ticket. \n\n"
message += f"Ticket: {structured_response.ticket}\n"
message += f"Priority: {structured_response.priority}\n\n"

messages = [{"role": "user", "content": message}]

In [42]:
# structured response
ticket_response = client.chat.completions.parse(
    model="gpt-4.1-nano",
    messages=messages,
    response_format=TicketResponse
)

ticket_response = ticket_response.choices[0].message.parsed
display(Markdown(f"### Response:\n{ticket_response.response}"))
display(Markdown(f"### Resolution Time:\n{ticket_response.resolution_time}"))

### Response:
Dear Customer,

Thank you for reaching out regarding the licensing error preventing activation of your software. We understand the urgency and apologize for the inconvenience caused.

Please follow these steps to resolve the issue:

1. Ensure that your internet connection is stable.
2. Verify that you have entered the license key correctly, without any typos.
3. Run the software as an administrator.
4. Temporarily disable any firewall or antivirus software that might be blocking activation.
5. Restart your computer and attempt activation again.

If the problem persists, please provide us with your license key and the exact error message you received. We may need to generate a new license or investigate further.

Our support team is committed to resolving this issue promptly. Thank you for your patience.

Best regards,
Customer Support Team

### Resolution Time:
Within 24 hours

## Lets evaluate our response

In [43]:
# messages list
message = "You are to evaluate the proposed resolution for the following customer support ticket. "
message += "You will determine if the proposed resolution is appropriate for the ticket and priority level. "
message += "tickets\n\n"
message += f"Ticket: {structured_response.ticket}\n"
message += f"Priority: {structured_response.priority}\n\n"
message += f"Proposed Resolution: {ticket_response.response}\n"
message += f"Proposed Resolution {ticket_response.resolution_time}\n\n"

messages = [{"role": "user", "content": message}]

In [44]:
messages

[{'role': 'user',
  'content': 'You are to evaluate the proposed resolution for the following customer support ticket. You will determine if the proposed resolution is appropriate for the ticket and priority level. tickets\n\nTicket: The customer is unable to activate their purchased software due to a licensing error.\nPriority: High\n\nProposed Resolution: Dear Customer,\n\nThank you for reaching out regarding the licensing error preventing activation of your software. We understand the urgency and apologize for the inconvenience caused.\n\nPlease follow these steps to resolve the issue:\n\n1. Ensure that your internet connection is stable.\n2. Verify that you have entered the license key correctly, without any typos.\n3. Run the software as an administrator.\n4. Temporarily disable any firewall or antivirus software that might be blocking activation.\n5. Restart your computer and attempt activation again.\n\nIf the problem persists, please provide us with your license key and the exa

In [45]:
# evaluate response
evaluator_response = client.chat.completions.parse(
    model="gpt-4.1-nano",
    messages=messages,
    response_format=TicketEvaluation
)

evaluator_response = evaluator_response.choices[0].message.parsed
display(Markdown(f"### Passed:\n{evaluator_response.passed}"))
display(Markdown(f"### Feedback:\n{evaluator_response.feedback}"))

### Passed:
True

### Feedback:
The proposed resolution is appropriate for the high-priority ticket. It provides clear, actionable steps to troubleshoot the licensing error promptly, addresses the customer's urgency, and offers further assistance if needed. The tone is professional and empathetic, aligning well with the urgency of the issue.

<div style="border-radius:16px;background:#1e2a1e;margin:1em 0;padding:1em 1em 1em 3em;color:#eceff4;position:relative;box-shadow:0 6px 16px rgba(0,0,0,.4)">
  <b style="color:#a3be8c;font-size:1.25em">Your Challenge:</b>
  <ul style="margin:.6em 0 0;padding-left:1.2em;line-height:1.6">
    <li>Hey everyone! Ready to flex those agentic muscles? 🎉 Build a workflow just like the ticket system above, but for <b>product reviews</b>!</li>
    <li>Your workflow should:
      <ul>
        <li>Generate a product review (think: electronics, books, or your favorite kitchen gadget)</li>
        <li>Respond to the review (company reply, moderation, or a witty bot response)</li>
        <li>Evaluate the response (is it helpful, polite, and on point?)</li>
      </ul>
    </li>
    <li>Use structured outputs and Pydantic models for each step, just like we did above.</li>
    <li>Include an evaluator step to assess the quality of the response.</li>
    <li>Here’s a suggested workflow to get your creative gears turning:</li>
  </ul>
  <div style="position:absolute;top:-.8em;left:-.8em;width:2.4em;height:2.4em;border-radius:50%;background:#a3be8c;color:#2e3440;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:1.2em">💪</div>
</div>

### Suggested Workflow

```mermaid
graph LR
    A[Generate Review] --> B[Respond to Review]
    B --> C[Evaluate Response]
    C --> B
    C --> D[Final Output]
```

Try to use structured outputs and Pydantic models for each step, just like in the notebook above. Include an evaluator step to assess the quality of the response.

In [46]:
# classes
from pydantic import BaseModel

class ReviewGenerated(BaseModel):
    item: str
    review: str
    recommendation: str
 
class ReviewEvaluation(BaseModel):
    feedback: str
    goodreview: bool


In [47]:
# messages list
user_message = "Generate a brief review of the book. War and Peace."
user_message += "Consider the historical issues. "
user_message += "Please ensure the review is varied."

messages = [{"role": "user", "content": user_message}]

In [48]:
# structured response, will generate a "random" review that fits the ReviewGenerated schema
structured_response = client.chat.completions.parse(
    model="gpt-4.1-nano",
    messages=messages,
    # new, specify the response format
    response_format=ReviewGenerated
)
# new, access message.parsed instead of message.content
structured_response = structured_response.choices[0].message.parsed
display(Markdown(f"### Structured Review:\n{structured_response}"))

### Structured Review:
item="'War and Peace' by Leo Tolstoy is an epic masterpiece that intricately explores the complexities of Russian society during the Napoleonic Wars. Its detailed depiction of historical events, combined with profound philosophical insights, offers a multifaceted view of human nature and the impact of war. While some readers might find its extensive character list and detailed descriptions challenging, the novel's rich themes and historical accuracy make it a monumental work in literature." review="Tolstoy's 'War and Peace' masterfully intertwines personal stories with the sweeping tide of history, providing a nuanced perspective on the sociopolitical upheavals of early 19th-century Russia. Its portrayal of the Napoleonic invasion highlights the chaos and tragedy of war, yet also celebrates the resilience of the human spirit. The novel's reflection on fate, free will, and the nature of history remains profoundly relevant. However, its complex narrative and philosophical discourse may require patient reading but reward those willing to delve into its depths." recommendation='Highly recommended for readers interested in historical fiction, Russian literature, or philosophical explorations of war. Due to its length and depth, it suits those seeking a deeply enriching reading experience.'

In [49]:
structured_response.item

"'War and Peace' by Leo Tolstoy is an epic masterpiece that intricately explores the complexities of Russian society during the Napoleonic Wars. Its detailed depiction of historical events, combined with profound philosophical insights, offers a multifaceted view of human nature and the impact of war. While some readers might find its extensive character list and detailed descriptions challenging, the novel's rich themes and historical accuracy make it a monumental work in literature."

In [50]:
structured_response.review

"Tolstoy's 'War and Peace' masterfully intertwines personal stories with the sweeping tide of history, providing a nuanced perspective on the sociopolitical upheavals of early 19th-century Russia. Its portrayal of the Napoleonic invasion highlights the chaos and tragedy of war, yet also celebrates the resilience of the human spirit. The novel's reflection on fate, free will, and the nature of history remains profoundly relevant. However, its complex narrative and philosophical discourse may require patient reading but reward those willing to delve into its depths."

In [51]:
structured_response.recommendation

'Highly recommended for readers interested in historical fiction, Russian literature, or philosophical explorations of war. Due to its length and depth, it suits those seeking a deeply enriching reading experience.'

In [52]:
# messages list
message = "You are to evaluate the review provided. \n\n"
message += f"Review: {structured_response.item}\n"
message += f"Evaluation: {structured_response.recommendation}\n\n"

messages = [{"role": "user", "content": message}]

In [53]:
# evaluate review
evaluator_response = client.chat.completions.parse(
    model="gpt-4.1-nano",
    messages=messages,
    response_format=ReviewEvaluation
)

evaluator_response = evaluator_response.choices[0].message.parsed
display(Markdown(f"### Passed:\n{evaluator_response.goodreview}"))
display(Markdown(f"### Feedback:\n{evaluator_response.feedback}"))

### Passed:
True

### Feedback:
The review provides a thorough and positive assessment of 'War and Peace,' highlighting its historical accuracy, thematic depth, and literary significance. It also acknowledges potential challenges for some readers, offering a balanced perspective. Overall, it effectively conveys why the novel is a monumental work and who might appreciate it most.